# Gaussian Filter

**Dataset**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz  
**Subject**: 1

---

## Overview

The Gaussian filter uses a bell-shaped kernel where distant samples contribute less. Larger `sigma` produces more smoothing and spreads each sample's influence over more neighbors.

## What you should expect to see

- sigma=2: light smoothing preserves details
- sigma=5: moderate smoothing removes fast noise
- sigma=10: strong smoothing noticeably smooths the signal

## Key parameters

| Parameter | Value | Meaning |
| --- | --- | --- |
| Channel | P4 | Parietal region |
| Sampling rate | 200 Hz | One sample every 5 ms |
| sigma | 2, 5, 10 | Gaussian kernel width |
| Plotted samples | 5000 | First 25 seconds |


## 1. Install dependencies


In [ ]:
!pip install scipy numpy plotly wfdb


## 2. Clone the resources repo and download one subject

We download only one subject (`--subjects 1`) to speed up the experiment in Colab.


In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


## 3. Load the EEG signal

We load subject 1, experiment 1, session 2, channel **P4** (parietal region).


In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 200  # Sampling rate (Hz)

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')


## 4. Apply the filter

We use `scipy.ndimage.gaussian_filter1d` which applies a Gaussian kernel along one axis. `sigma` controls the kernel width.


In [ ]:
from scipy.ndimage import gaussian_filter1d

sigmas = [2, 5, 10]
filtered = {}
for s in sigmas:
    filtered[s] = gaussian_filter1d(channel_data, sigma=s)
print(f'Applied Gaussian filter with sigmas: {sigmas}')


## 5. Interactive plot

**What to look for:**

- sigma=2: details remain clear
- sigma=5: fast noise removed, signal stays coherent
- sigma=10: strong smoothing hides fine details
- Compare the three windows to see the effect of `sigma`


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

n_plot = min(5000, len(channel_data))
t_sec = timestamps[:n_plot] / 1000.0

fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                    subplot_titles=('Original (P4)',
                                    'Gaussian (sigma=2)',
                                    'Gaussian (sigma=5)',
                                    'Gaussian (sigma=10)'))
fig.add_trace(go.Scatter(x=t_sec, y=channel_data[:n_plot], name='Raw',
                         line=dict(color='gray', width=0.5)), row=1, col=1)
for i, s in enumerate(sigmas, start=2):
    fig.add_trace(go.Scatter(x=t_sec, y=filtered[s][:n_plot],
                             name=f'sigma={s}', line=dict(width=0.5)), row=i, col=1)
fig.update_layout(height=900, title_text='Gaussian Filter - Channel P4',
                  xaxis4_title='Time (s)', showlegend=False)
fig.show()


## What did we learn?

- The Gaussian filter uses a bell-shaped kernel where distant samples contribute less
- Larger `sigma` produces stronger smoothing
- The Gaussian filter preserves signal shape better than the moving average
- It is well suited for reducing noise before frequency analysis
